<h4> Lanturile Markov avand 1 stare </h4>

<h1>Importuri</h1>

In [ ]:
import numpy as np
import pandas as pd
import os
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import random

<h1>Citirea datelor</h1>

In [ ]:
def read_all(path):
    txt = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line == '----------':
                break
            if line != '':
                txt.append(line)
    return txt

path = "D:/FACULTATE/SEM4/AI/Lab8/data/proverbe.txt"
stories = read_all(path)
print("Num?rul de linii =", len(stories))

<h1>Cleaning DataSet</h1>

In [ ]:
def clean_txt(txt):
    cleaned_txt = []
    for line in txt:
        line = line.lower() 
        
        line = re.sub(r"[,.\"\'!@#$%^&*(){}?/;`~:<>+=-\\]", "", line)
        
        tokens = word_tokenize(line)  # Tokenizam 
        words = [word for word in tokens if word.isalpha()]  # Alegem doar cuvintele
        cleaned_txt.extend(words)  
    return cleaned_txt

cleaned_stories = clean_txt(stories)
print("Num?rul de cuvinte =", len(cleaned_stories))
print(cleaned_stories[:50])

<h1>Crearea modelului Markov</h1>

In [ ]:
def make_markov_model(cleaned_stories, n_gram):
    markov_model = {}
    for i in range(len(cleaned_stories)-n_gram):
        curr_state, next_state = "", ""
        for j in range(n_gram):
            curr_state += cleaned_stories[i+j] + " "
            next_state += cleaned_stories[i+j+n_gram] + " " if i + j + n_gram < len(cleaned_stories) else ""
        curr_state = curr_state[:-1]
        next_state = next_state[:-1]
        if curr_state not in markov_model:
            markov_model[curr_state] = {}
            markov_model[curr_state][next_state] = 1
        else:
            if next_state in markov_model[curr_state]:
                markov_model[curr_state][next_state] += 1
            else:
                markov_model[curr_state][next_state] = 1
    
    # calculating transition probabilities
    for curr_state, transition in markov_model.items():
        total = sum(transition.values())
        for state, count in transition.items():
            markov_model[curr_state][state] = count/total
        
    return markov_model

In [ ]:
markov_model = make_markov_model(cleaned_stories, 1)

In [ ]:
print("number of states = ", len(markov_model.keys()))

In [ ]:
print("All possible transitions from 'the game' state: \n")
print(markov_model['face'])

<h1>Generarea de proverbe</h1>

In [ ]:
def generate_story(markov_model, limit, start):
    n = 0
    curr_state = start
    next_state = None
    story = ""
    story+=curr_state+" "
    while n<limit:
        next_state = random.choices(list(markov_model[curr_state].keys()),
                                    list(markov_model[curr_state].values()))
        
        curr_state = next_state[0]
        story+=curr_state+" "
        n+=1
    return story

<h1>Utilizare</h1>

In [ ]:
for i in range(40):
    print(str(i)+". ", generate_story(markov_model, 8 ,"face"))